In [ ]:
import json
from pathlib import Path
import pandas as pd

# -------------------------------------------------------------------
# 1. 파일별 전처리 함수 정의
# -------------------------------------------------------------------


def process_product_csv(file_path: Path):
    """끝이 product인 CSV 파일 처리"""
    print(f"📦 [Product] 처리 시작: {file_path.name}")
    target_cols = ["쇼핑몰명", "대분류", "중분류", "소분류", "상품명"]

    try:
        df = pd.read_csv(file_path, usecols=target_cols)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, usecols=target_cols, encoding="cp949")

    # 💡 카테고리 ' > ' 연결 로직 (빈 값 자동 제외)
    cat_cols = ["대분류", "중분류", "소분류"]

    def join_categories(row):
        cats = [
            str(val).strip()
            for val in row
            if pd.notna(val) and str(val).strip()
        ]
        return " > ".join(cats)

    product_cat = df[cat_cols].apply(join_categories, axis=1)

    result_df = pd.DataFrame(
        {
            "brand_name": df["쇼핑몰명"].astype(str).str.strip(),
            "product_name": df["상품명"].astype(str).str.strip(),
            "product_cat": product_cat,
        }
    )

    print(f"  └ 완료 ({len(result_df)}행)")
    return result_df


def process_image_csv(file_path: Path):
    """끝이 image인 CSV 파일 처리"""
    print(f"🖼️ [Image] 처리 시작: {file_path.name}")
    target_cols = ["상품명", "상세이미지순번", "상세이미지주소링크"]

    try:
        df = pd.read_csv(file_path, usecols=target_cols)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, usecols=target_cols, encoding="cp949")

    result_df = pd.DataFrame(
        {
            "product_name": df["상품명"].astype(str).str.strip(),
            "image_sequence": df["상세이미지순번"],
            "image_url": df["상세이미지주소링크"].astype(str).str.strip(),
        }
    )

    print(f"  └ 완료 ({len(result_df)}행)")
    return result_df


def process_jl_jsonl(file_path: Path):
    """끝이 jl인 JSONL 파일 처리"""
    print(f"📄 [JSONL] 처리 시작: {file_path.name}")

    jsonl_dict = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            p_name = item.get("상품명")
            p_jsonld = item.get("product_jsonld")

            if p_name:
                p_name_clean = str(p_name).strip()
                jsonl_dict[p_name_clean] = p_jsonld

    print(f"  └ 완료 (매핑 항목: {len(jsonl_dict)}개)")
    return jsonl_dict


# -------------------------------------------------------------------
# 2. 메인 파이프라인 함수
# -------------------------------------------------------------------


def run_pipeline(target_directory: str = "."):
    dir_path = Path(target_directory)

    product_dfs = []
    image_dfs = []
    combined_jsonl_dict = {}

    for file_path in dir_path.rglob("*"):
        if not file_path.is_file():
            continue

        if file_path.suffix == ".ipynb" or file_path.name.startswith("."):
            continue

        file_stem = file_path.stem
        parts = file_stem.split("_")
        file_type = parts[-1]

        # 1. Product CSV
        if file_type == "product" and file_path.suffix == ".csv":
            df = process_product_csv(file_path)
            product_dfs.append(df)

        # 2. Image CSV
        elif file_type == "image" and file_path.suffix == ".csv":
            df = process_image_csv(file_path)
            image_dfs.append(df)

        # 3. JSONL (jl)
        elif file_type == "jl" and file_path.suffix in [".jsonl", ".jl"]:
            jl_map = process_jl_jsonl(file_path)
            combined_jsonl_dict.update(jl_map)

    print("\n" + "=" * 50)

    total_product_df = (
        pd.concat(product_dfs, ignore_index=True) if product_dfs else None
    )
    total_image_df = (
        pd.concat(image_dfs, ignore_index=True) if image_dfs else None
    )

    # Product DataFrame 매칭 및 has_json 설정
    if total_product_df is not None:
        total_product_df["json_ld_contents"] = total_product_df[
            "product_name"
        ].map(combined_jsonl_dict)
        total_product_df["has_json_ld"] = total_product_df[
            "json_ld_contents"
        ].notna()

        matched_count = total_product_df["has_json"].sum()
        print(
            f"🎯 Product CSV & JSONL 매칭 완료: 총 {len(total_product_df)}개 중 {matched_count}개 매칭 성공 (has_json=True)"
        )

    return total_product_df, total_image_df


# -------------------------------------------------------------------
# 3. 실행 및 결과 확인
# -------------------------------------------------------------------
final_product_df, final_image_df = run_pipeline(".")

if final_product_df is not None:
    print("\n📦 [최종 Product DataFrame 예시]")
    display(final_product_df.head())

🖼️ [Image] 처리 시작: holly_blsh_image.csv
  └ 완료 (304행)
📄 [JSONL] 처리 시작: holly_blsh_jl.jsonl
  └ 완료 (매핑 항목: 20개)
📦 [Product] 처리 시작: holly_blsh_product.csv
  └ 완료 (20행)
🖼️ [Image] 처리 시작: holly_hd_image.csv
  └ 완료 (128행)
📄 [JSONL] 처리 시작: holly_hd_jl.jsonl
  └ 완료 (매핑 항목: 6개)
📦 [Product] 처리 시작: holly_hd_product.csv
  └ 완료 (6행)
🖼️ [Image] 처리 시작: holly_lknit_image.csv
  └ 완료 (408행)
📄 [JSONL] 처리 시작: holly_lknit_jl.jsonl
  └ 완료 (매핑 항목: 20개)
📦 [Product] 처리 시작: holly_lknit_product.csv
  └ 완료 (20행)
🖼️ [Image] 처리 시작: holly_ls_image.csv
  └ 완료 (164행)
📄 [JSONL] 처리 시작: holly_ls_jl.jsonl
  └ 완료 (매핑 항목: 13개)
📦 [Product] 처리 시작: holly_ls_product.csv
  └ 완료 (13행)
🖼️ [Image] 처리 시작: holly_lt_image.csv
  └ 완료 (483행)
📄 [JSONL] 처리 시작: holly_lt_jl.jsonl
  └ 완료 (매핑 항목: 20개)
📦 [Product] 처리 시작: holly_lt_product.csv
  └ 완료 (20행)
🖼️ [Image] 처리 시작: holly_mtm_image.csv
  └ 완료 (508행)
📄 [JSONL] 처리 시작: holly_mtm_jl.jsonl
  └ 완료 (매핑 항목: 18개)
📦 [Product] 처리 시작: holly_mtm_product.csv
  └ 완료 (18행)
🖼️ [Image] 처리 시작: holly_sknit_

,brand_name,product_name,product_cat,json_ld_contents,has_json
0,홀리,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True
1,홀리,[팔뚝삭제/허얇핏] 앤드리 도트 콩단추 반팔 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True
2,홀리,*7col [햇빛막아주는/여리!] 파파야 시스루 썸머 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True
3,홀리,[여리여리-/부유방부각X] 슈디 캉캉 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True
4,홀리,[여름내내/힙덮는] 먼데이 린넨 스트라이프 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True


In [4]:
# Image 테이블 상위 5개 행 확인
if final_image_df is not None:
    display(final_image_df.head(30))
else:
    print("Image 파일이 없거나 처리되지 않았습니다.")

,product_name,image_sequence,image_url
0,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,1,https://imgb.a-bly.com/data/editor/20260626_17...
1,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,2,https://imgb.a-bly.com/data/editor/20260626_17...
2,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,3,https://imgb.a-bly.com/data/editor/20260626_17...
3,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,4,https://imgb.a-bly.com/data/editor/20260626_17...
4,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,5,https://imgb.a-bly.com/data/editor/20260626_17...
5,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,6,https://imgb.a-bly.com/data/editor/20260626_17...
6,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,7,https://imgb.a-bly.com/data/editor/20260626_17...
7,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,8,https://imgb.a-bly.com/data/editor/20260626_17...
8,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,9,https://imgb.a-bly.com/data/editor/20260626_17...
9,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,10,https://imgb.a-bly.com/data/editor/20260626_17...


In [10]:
final_product_df['has_json'].value_counts()

has_json
True    1659
Name: count, dtype: int64

In [6]:
# 1. Product 테이블에 포함된 브랜드 목록 및 브랜드별 상품 개수 확인
print("📦 Product 테이블 브랜드 목록:")
print(final_product_df['brand_name'].value_counts())

# 2. Image 테이블에 포함된 행 개수 확인 (필요한 경우)
if final_image_df is not None:
    print("\n🖼️ Image 테이블 총 행 개수:", len(final_image_df))

📦 Product 테이블 브랜드 목록:
brand_name
아이딜릭마켓    629
슈가파우더     404
자스민벨      281
홀리        165
보통의날      100
미니포에       80
Name: count, dtype: int64

🖼️ Image 테이블 총 행 개수: 29149


In [21]:
final_product_df.head(30)

,brand_name,product_name,product_cat,json_ld_contents,has_json,brand_type
0,홀리,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
1,홀리,[팔뚝삭제/허얇핏] 앤드리 도트 콩단추 반팔 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
2,홀리,*7col [햇빛막아주는/여리!] 파파야 시스루 썸머 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
3,홀리,[여리여리-/부유방부각X] 슈디 캉캉 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
4,홀리,[여름내내/힙덮는] 먼데이 린넨 스트라이프 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
5,홀리,[여리글램!/분위기보장] 미뇽 퍼프 프릴 반팔 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
6,홀리,[캡내장/글램여리🪽] 퓨어 캡 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
7,홀리,[여름필수!/초경량] 에어 텐셀 썸머 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
8,홀리,[여리루즈!/시어서커] 시오 썸머 스트라이프 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
9,홀리,[투웨이!/문의폭주] 바닐라 퍼프 크롭 반팔 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인


In [9]:
final_image_df.head()

,product_name,image_sequence,image_url
0,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,1,https://imgb.a-bly.com/data/editor/20260626_17...
1,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,2,https://imgb.a-bly.com/data/editor/20260626_17...
2,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,3,https://imgb.a-bly.com/data/editor/20260626_17...
3,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,4,https://imgb.a-bly.com/data/editor/20260626_17...
4,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,5,https://imgb.a-bly.com/data/editor/20260626_17...


In [11]:
final_product_df['brand_type'] = '소상공인'
final_product_df.head()

,brand_name,product_name,product_cat,json_ld_contents,has_json,brand_type
0,홀리,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
1,홀리,[팔뚝삭제/허얇핏] 앤드리 도트 콩단추 반팔 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
2,홀리,*7col [햇빛막아주는/여리!] 파파야 시스루 썸머 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
3,홀리,[여리여리-/부유방부각X] 슈디 캉캉 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
4,홀리,[여름내내/힙덮는] 먼데이 린넨 스트라이프 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인


In [ ]:
db_df = final_product_df.copy()

# 컬럼명 맞추기 (has_json -> has_json_ld)
db_df = db_df.rename(columns={"has_json": "has_json_ld"})

In [14]:
db_df.head()

,brand_name,product_name,product_cat,json_ld_contents,has_json_ld,brand_type
0,홀리,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
1,홀리,[팔뚝삭제/허얇핏] 앤드리 도트 콩단추 반팔 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
2,홀리,*7col [햇빛막아주는/여리!] 파파야 시스루 썸머 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
3,홀리,[여리여리-/부유방부각X] 슈디 캉캉 나시 블라우스,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인
4,홀리,[여름내내/힙덮는] 먼데이 린넨 스트라이프 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{'@context': 'https://schema.org', '@type': '...",True,소상공인


In [15]:
def convert_to_json_str(val):
    if pd.isna(val) or val is None:
        return None
    if isinstance(val, (dict, list)):
        return json.dumps(val, ensure_ascii=False)
    return str(val)


db_df["json_ld_contents"] = db_df["json_ld_contents"].apply(
    convert_to_json_str
)

In [16]:
db_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1659 entries, 0 to 1658
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   brand_name        1659 non-null   str  
 1   product_name      1659 non-null   str  
 2   product_cat       1659 non-null   str  
 3   json_ld_contents  1659 non-null   str  
 4   has_json_ld       1659 non-null   bool 
 5   brand_type        1659 non-null   str  
dtypes: bool(1), str(5)
memory usage: 6.6 MB


In [19]:
db_df["text_contents"] = ""
db_df.head()

,brand_name,product_name,product_cat,json_ld_contents,has_json_ld,brand_type,text_contents
0,홀리,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,상의 > 블라우스&셔츠,"[{""@context"": ""https://schema.org"", ""@type"": ""...",True,소상공인,
1,홀리,[팔뚝삭제/허얇핏] 앤드리 도트 콩단추 반팔 블라우스,상의 > 블라우스&셔츠,"[{""@context"": ""https://schema.org"", ""@type"": ""...",True,소상공인,
2,홀리,*7col [햇빛막아주는/여리!] 파파야 시스루 썸머 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{""@context"": ""https://schema.org"", ""@type"": ""...",True,소상공인,
3,홀리,[여리여리-/부유방부각X] 슈디 캉캉 나시 블라우스,상의 > 블라우스&셔츠,"[{""@context"": ""https://schema.org"", ""@type"": ""...",True,소상공인,
4,홀리,[여름내내/힙덮는] 먼데이 린넨 스트라이프 루즈핏 셔츠,상의 > 블라우스&셔츠,"[{""@context"": ""https://schema.org"", ""@type"": ""...",True,소상공인,


In [17]:
!pip install pymysql sqlalchemy
import json
import pymysql
from sqlalchemy import create_engine


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# -------------------------------------------------------------------
# 2. MySQL DB 연결 설정
# -------------------------------------------------------------------
# 본인의 MySQL 접속 정보를 입력하세요 (user, password, host, port)
db_user = "root"
db_password = "1234"  # DB 비밀번호
db_host = "localhost"  # DB 호스트 (ex: localhost 또는 IP)
db_port = "3306"
db_name = "geo_db"  # DB명

# SQLAlchemy Engine 생성 (PyMySQL 드라이버 사용)
engine_url = (
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)
engine = create_engine(engine_url)


# -------------------------------------------------------------------
# 3. raw data table에 데이터 적재 (Insert)
# -------------------------------------------------------------------
table_name = "raw_data_table"  # 💡 실제 생성해두신 DB 테이블명을 입력하세요!

try:
    # if_exists='append': 기존 테이블 구조 유지하면서 아래에 데이터 추가
    # index=False: DataFrame의 index 숫자는 DB 컬럼으로 넣지 않음
    db_df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000,  # 대용량 데이터를 위해 1,000건씩 분할 전송
    )
    print(f"🚀 {len(db_df)}건의 데이터가 '{table_name}' 테이블에 성공적으로 적재되었습니다!")

except Exception as e:
    print(f"❌ DB 적재 중 오류 발생: {e}")

🚀 1659건의 데이터가 'raw_data_table' 테이블에 성공적으로 적재되었습니다!


In [23]:
final_image_df.head()

,product_name,image_sequence,image_url
0,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,1,https://imgb.a-bly.com/data/editor/20260626_17...
1,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,2,https://imgb.a-bly.com/data/editor/20260626_17...
2,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,3,https://imgb.a-bly.com/data/editor/20260626_17...
3,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,4,https://imgb.a-bly.com/data/editor/20260626_17...
4,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,5,https://imgb.a-bly.com/data/editor/20260626_17...


In [27]:
# final_product_df에 있는 brand_name과 product_name 관계를 이용해 final_image_df에 brand_name 매칭
brand_map = final_product_df[['product_name', 'brand_name']].drop_duplicates()
final_image_df = pd.merge(final_image_df, brand_map, on='product_name', how='left')

In [28]:
final_image_df.head()

,product_name,image_sequence,image_url,brand_name_x,brand_name_y
0,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,1,https://imgb.a-bly.com/data/editor/20260626_17...,홀리,홀리
1,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,2,https://imgb.a-bly.com/data/editor/20260626_17...,홀리,홀리
2,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,3,https://imgb.a-bly.com/data/editor/20260626_17...,홀리,홀리
3,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,4,https://imgb.a-bly.com/data/editor/20260626_17...,홀리,홀리
4,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,5,https://imgb.a-bly.com/data/editor/20260626_17...,홀리,홀리


In [30]:
final_image_df.drop(columns='brand_name_y', inplace=True)

In [32]:
final_image_df.rename(columns={'brand_name_x':'brand_name'}, inplace=True)
final_image_df

,product_name,image_sequence,image_url,brand_name
0,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,1,https://imgb.a-bly.com/data/editor/20260626_17...,홀리
1,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,2,https://imgb.a-bly.com/data/editor/20260626_17...,홀리
2,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,3,https://imgb.a-bly.com/data/editor/20260626_17...,홀리
3,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,4,https://imgb.a-bly.com/data/editor/20260626_17...,홀리
4,[허리잘록!/여름데이트🌿] 플래닝 핀턱 스트링 나시 블라우스,5,https://imgb.a-bly.com/data/editor/20260626_17...,홀리
...,...,...,...,...
29144,"Made [빈티지🤍/오버핏!] 네오 윈드브레이커 오버핏 고프코어룩 바람막이 점퍼,집...",27,https://imgb.a-bly.com/data/editor/20250310_17...,슈가파우더
29145,"Made [빈티지🤍/오버핏!] 네오 윈드브레이커 오버핏 고프코어룩 바람막이 점퍼,집...",28,https://imgb.a-bly.com/data/editor/20250310_17...,슈가파우더
29146,"Made [빈티지🤍/오버핏!] 네오 윈드브레이커 오버핏 고프코어룩 바람막이 점퍼,집...",29,https://imgb.a-bly.com/data/editor/20250310_17...,슈가파우더
29147,"Made [빈티지🤍/오버핏!] 네오 윈드브레이커 오버핏 고프코어룩 바람막이 점퍼,집...",30,https://imgb.a-bly.com/data/editor/20250310_17...,슈가파우더


In [35]:
# -------------------------------------------------------------------
# 3. DB의 raw_data_table에서 page_id 가져와서 매칭 (1:1 정확한 매칭)
# -------------------------------------------------------------------
query = "SELECT page_id, brand_name, product_name FROM raw_data_table"
product_id_df = pd.read_sql(query, con=engine)

image_db_df = final_image_df.copy()

# brand_name + product_name 두 개를 기준(Key)으로 page_id 조인
image_db_df = pd.merge(
    image_db_df, product_id_df, on=["brand_name", "product_name"], how="inner"
)

# -------------------------------------------------------------------
# 4. NOT NULL 컬럼 기본값 채우기 및 정렬
# -------------------------------------------------------------------
image_db_df["image_text"] = ""
image_db_df["has_alt"] = False
image_db_df["alt_contents"] = None

# DB 컬럼 순서 및 규격에 맞게 추출
target_columns = [
    "page_id",
    "brand_name",
    "image_sequence",
    "image_text",
    "image_url",
    "has_alt",
    "alt_contents",
]
image_db_df = image_db_df[target_columns]

# (page_id, image_sequence) 중복 사전에 방지
image_db_df = image_db_df.drop_duplicates(
    subset=["page_id", "image_sequence"], keep="first"
)

# -------------------------------------------------------------------
# 5. DB 적재
# -------------------------------------------------------------------
image_table_name = "image_data_table"

try:
    image_db_df.to_sql(
        name=image_table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000,
    )
    print(
        f"🚀 {len(image_db_df)}건의 이미지 데이터가 '{image_table_name}' 테이블에 성공적으로 적재되었습니다!"
    )

except Exception as e:
    print(f"❌ DB 적재 중 오류 발생: {e}")

🚀 41335건의 이미지 데이터가 'image_data_table' 테이블에 성공적으로 적재되었습니다!


In [5]:
import pandas as pd
import glob
import os

# 1. 파일 및 경로 설정
target_csv_path = "jsonld 누락사항_인코딩.csv"          # SQL에서 내보내고 인코딩 정제한 CSV
output_csv_path = 'jsonld_X_updated.csv'  # 최종 결과물 저장 파일명

# 💡 cj 폴더 경로 지정
product_dir = 'cj'

# 2. project_db/cj 폴더 내의 모든 *product*.csv 파일 검색
search_pattern = os.path.join(product_dir, '*product*.csv')
product_files = glob.glob(search_pattern)

# 만약 확장자가 대문자 .CSV일 가능성 대비
if not product_files:
    product_files = glob.glob(os.path.join(product_dir, '*product*.CSV'))

print(f"🔍 탐색 경로: {os.path.abspath(product_dir)}")
print(f"📦 찾은 product 파일 목록 ({len(product_files)}개): {product_files}")

# ⚠️ 예외 처리 (파일 탐색 확인)
if len(product_files) == 0:
    raise FileNotFoundError(f"❌ '{product_dir}' 경로에서 *product*.csv 파일들을 찾지 못했습니다!")

# 여러 product CSV를 하나의 데이터프레임으로 수집
df_list = []
for file in product_files:
    try:
        temp_df = pd.read_csv(file, encoding='utf-8-sig')
    except UnicodeDecodeError:
        temp_df = pd.read_csv(file, encoding='cp949')
    
    df_list.append(temp_df)

# 모든 product 데이터 합치기 및 중복 제거
df_all_products = pd.concat(df_list, ignore_index=True)
df_all_products = df_all_products.drop_duplicates(subset=['상품명'])

# 3. 통합 매핑 사전(Dictionary) 만들기
brand_map = df_all_products.set_index('상품명')['브랜드'].to_dict()

# 4. Target CSV 불러오기 및 brand_name 업데이트
try:
    df_target = pd.read_csv(target_csv_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_target = pd.read_csv(target_csv_path, encoding='cp949')

# 매칭되는 brand_name으로 변경 (매칭 안 되는 건 기존 'CJ' 유지)
df_target['brand_name'] = df_target['product_name'].map(brand_map).fillna(df_target['brand_name'])

# 5. UTF-8 BOM(utf-8-sig) 인코딩으로 저장 (엑셀 오픈 시 안 깨짐)
df_target.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"✅ 총 {len(brand_map)}개 상품 매핑 완료! 파일 저장: {output_csv_path}")

🔍 탐색 경로: c:\Users\hkssn\바탕 화면\multicam\project_db\cj
📦 찾은 product 파일 목록 (64개): ['cj\\cjm_cd_product.csv', 'cj\\cjm_ct_product.csv', 'cj\\cjm_dp_product.csv', 'cj\\cjm_hd_product.csv', 'cj\\cjm_jk_product.csv', 'cj\\cjm_jm_product.csv', 'cj\\cjm_jsuit_product.csv', 'cj\\cjm_knit_product.csv', 'cj\\cjm_leg_product.csv', 'cj\\cjm_lp_product.csv', 'cj\\cjm_lsh_product.csv', 'cj\\cjm_lt_product.csv', 'cj\\cjm_ms_product.csv', 'cj\\cjm_mtm_product.csv', 'cj\\cjm_ovest_product.csv', 'cj\\cjm_pad_product.csv', 'cj\\cjm_psuit_product.csv', 'cj\\cjm_ptr_product.csv', 'cj\\cjm_slt_product.csv', 'cj\\cjm_sp_product.csv', 'cj\\cjm_ssh_product.csv', 'cj\\cjm_ssuit_product.csv', 'cj\\cjm_str_product.csv', 'cj\\cjm_st_product.csv', 'cj\\cjm_tct_product.csv', 'cj\\cjm_ttr_product.csv', 'cj\\cjm_vest_product.csv', 'cj\\cjm_wind_product.csv', 'cj\\cjw_blouse_product.csv', 'cj\\cjw_btr_product.csv', 'cj\\cjw_cd_product.csv', 'cj\\cjw_ct_product.csv', 'cj\\cjw_dp_product.csv', 'cj\\cjw_hd_product.csv', 'cj

In [17]:
import glob
import pandas as pd

# 1. 모든 이미지 CSV 불러오기
file_paths = glob.glob("./loofon/*_image.csv") # 본인의 경로
all_dfs = [pd.read_csv(p) for p in file_paths]
total_df = pd.concat(all_dfs, ignore_index=True)

# 2. PHASE 3처럼 상품명 공백 정제
total_df["상품명"] = total_df["상품명"].astype(str).str.strip()

# 3. [상품명 + 이미지 순서] 기준으로 겹치는 365개 뽑기 (두 번째 중복분만)
dup_365 = total_df[total_df.duplicated(subset=["상품명", "상세이미지순번"])]

print(f"🎯 [상품명 + image_sequence 기준 중복 건수]: {len(dup_365)}건")
print("\n👇 실제로 버려진 365개 중 상위 5개 샘플:")
print(dup_365[["상품명", "상세이미지순번"]].to_string(index=False))

🎯 [상품명 + image_sequence 기준 중복 건수]: 365건

👇 실제로 버려진 365개 중 상위 5개 샘플:
             상품명  상세이미지순번
    니스 하이넥 후드 니트        1
    니스 하이넥 후드 니트        2
    니스 하이넥 후드 니트        3
    니스 하이넥 후드 니트        4
    니스 하이넥 후드 니트        5
    니스 하이넥 후드 니트        6
    니스 하이넥 후드 니트        7
    니스 하이넥 후드 니트        8
    니스 하이넥 후드 니트        9
    니스 하이넥 후드 니트       10
    니스 하이넥 후드 니트       11
    니스 하이넥 후드 니트       12
    니스 하이넥 후드 니트       13
    니스 하이넥 후드 니트       14
    니스 하이넥 후드 니트       15
    니스 하이넥 후드 니트       16
    니스 하이넥 후드 니트       17
    니스 하이넥 후드 니트       18
    니스 하이넥 후드 니트       19
    니스 하이넥 후드 니트       20
    니스 하이넥 후드 니트       21
    니스 하이넥 후드 니트       22
    니스 하이넥 후드 니트       23
    니스 하이넥 후드 니트       24
    니스 하이넥 후드 니트       25
    니스 하이넥 후드 니트       26
    니스 하이넥 후드 니트       27
    니스 하이넥 후드 니트       28
    니스 하이넥 후드 니트       29
    니스 하이넥 후드 니트       30
    니스 하이넥 후드 니트       31
    니스 하이넥 후드 니트       32
    니스 하이넥 후드 니트       33
    니스 하이넥 후드 니트       34
    니스 하이넥 후드 니트      

In [1]:
import pandas as pd
import glob
import os

# 1. 파일 및 경로 설정
target_csv_path = "image_text_누락.csv"          # SQL에서 내보내고 인코딩 정제한 CSV
output_csv_path = 'imagetext_X_updated.csv'  # 최종 결과물 저장 파일명

# 💡 cj 폴더 경로 지정
product_dir = 'cj'

# 2. project_db/cj 폴더 내의 모든 *product*.csv 파일 검색
search_pattern = os.path.join(product_dir, '*product*.csv')
product_files = glob.glob(search_pattern)

# 만약 확장자가 대문자 .CSV일 가능성 대비
if not product_files:
    product_files = glob.glob(os.path.join(product_dir, '*product*.CSV'))

print(f"🔍 탐색 경로: {os.path.abspath(product_dir)}")
print(f"📦 찾은 product 파일 목록 ({len(product_files)}개): {product_files}")

# ⚠️ 예외 처리 (파일 탐색 확인)
if len(product_files) == 0:
    raise FileNotFoundError(f"❌ '{product_dir}' 경로에서 *product*.csv 파일들을 찾지 못했습니다!")

# 여러 product CSV를 하나의 데이터프레임으로 수집
df_list = []
for file in product_files:
    try:
        temp_df = pd.read_csv(file, encoding='utf-8-sig')
    except UnicodeDecodeError:
        temp_df = pd.read_csv(file, encoding='cp949')
    
    df_list.append(temp_df)

# 모든 product 데이터 합치기 및 중복 제거
df_all_products = pd.concat(df_list, ignore_index=True)
df_all_products = df_all_products.drop_duplicates(subset=['상품명'])

# 3. 통합 매핑 사전(Dictionary) 만들기
brand_map = df_all_products.set_index('상품명')['브랜드'].to_dict()

# 4. Target CSV 불러오기 및 brand_name 업데이트
try:
    df_target = pd.read_csv(target_csv_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_target = pd.read_csv(target_csv_path, encoding='cp949')

# 매칭되는 brand_name으로 변경 (매칭 안 되는 건 기존 'CJ' 유지)
df_target['brand_name'] = df_target['product_name'].map(brand_map).fillna(df_target['brand_name'])

# 5. UTF-8 BOM(utf-8-sig) 인코딩으로 저장 (엑셀 오픈 시 안 깨짐)
df_target.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"✅ 총 {len(brand_map)}개 상품 매핑 완료! 파일 저장: {output_csv_path}")

🔍 탐색 경로: c:\Users\hkssn\바탕 화면\multicam\project_db\cj
📦 찾은 product 파일 목록 (64개): ['cj\\cjm_cd_product.csv', 'cj\\cjm_ct_product.csv', 'cj\\cjm_dp_product.csv', 'cj\\cjm_hd_product.csv', 'cj\\cjm_jk_product.csv', 'cj\\cjm_jm_product.csv', 'cj\\cjm_jsuit_product.csv', 'cj\\cjm_knit_product.csv', 'cj\\cjm_leg_product.csv', 'cj\\cjm_lp_product.csv', 'cj\\cjm_lsh_product.csv', 'cj\\cjm_lt_product.csv', 'cj\\cjm_ms_product.csv', 'cj\\cjm_mtm_product.csv', 'cj\\cjm_ovest_product.csv', 'cj\\cjm_pad_product.csv', 'cj\\cjm_psuit_product.csv', 'cj\\cjm_ptr_product.csv', 'cj\\cjm_slt_product.csv', 'cj\\cjm_sp_product.csv', 'cj\\cjm_ssh_product.csv', 'cj\\cjm_ssuit_product.csv', 'cj\\cjm_str_product.csv', 'cj\\cjm_st_product.csv', 'cj\\cjm_tct_product.csv', 'cj\\cjm_ttr_product.csv', 'cj\\cjm_vest_product.csv', 'cj\\cjm_wind_product.csv', 'cj\\cjw_blouse_product.csv', 'cj\\cjw_btr_product.csv', 'cj\\cjw_cd_product.csv', 'cj\\cjw_ct_product.csv', 'cj\\cjw_dp_product.csv', 'cj\\cjw_hd_product.csv', 'cj